<a href="https://colab.research.google.com/github/nagamamo/machine_learning_for_finance/blob/main/11_1_monetary_policy_analysis_by_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**11-1. 大規模言語モデル（LLM）による中央銀行議事録の分析**

　2024年以降、大規模言語モデル（LLM）を用いる各国政府・中央銀行金議事録の分析を行う研究が多数発表されています。この研究の目的は、例えば中央銀行の金融政策を決定する会議の議事録を分析することで、金融市場の価格・金利形成の変化の芽をいち早く検出することです。関連研究では、2025年の国際通貨基金（IMF）ワーキング・ペーパー109号や2026年の*Journal of International Financial Markets, Institutions and Money*に掲載されたチェン教授（英クイーンメアリー大学）らの研究がその代表です。本例では、日本銀行の2026年7月の金融政策決定会合の議事録要旨を用い、出現頻度が高い経済用語のランキングを作成するソースコードを紹介します。

　まず、pdfファイルから文字データをテキストとして抽出するためのパッケージであるpypdf、そして抽出されたテキストの単語に細かく分割し、それぞれの品詞を判別する 形態素解析を行うjanomeをインストールします。

In [ ]:
pip install pypdf janome

　続いて、テキストに含まれる単語の回数をカウントするCounterをCollectionsから、pdfファイルからテキストを抽出するPdfReaderをpypdfから、インポートします。そして、日本語の場合は単語間にブランクがないため、janomeから単語と助詞を正確に分割するためのライブラリ、Tokenizerを読み込みます。

In [10]:
#[1]ライブラリのインポート
from collections import Counter
from pypdf import PdfReader
from janome.tokenizer import Tokenizer

　次のセル[2]からはpdfファイルをアップロードしてテキストを抽出します。この作業を実行するため、予め
https://www.boj.or.jp/mopo/mpmsche_minu/minu_2026/index.htm

から議事録要旨をダウンロードし、google colabの左側アイコンの下から2つ目の「ファイル」にアップロードします。このファイルはセッション・ストレージにアップロードされるため、終了後、同じ作業を実施する場合は逐次この作業が必要です。

　ファイルのアップロードが完了すれば、このPDFファイルからのテキスト抽出と、このテキスト内に含まれる単語の頻度カウント作業のワークフローを指定します。セル[2]では、（1）PDFファイルを読み込み、（2）このテキストを保存する変数を設定、（3）テキストを1頁ずつ処理実施、（4）テキストから抽出された文字データの保存、（5）次頁の抽出された文字データの追加、を指示しています。
　続いて（6）と（7）ではJanomeによりテキストを単語に分解して、カウントから除外する単語を指定します。（8）～（9）では、テキストから助詞を除外して名詞のみの文字データへ整形し、名詞か否か、除外リストに含まれるか否か、を判断してword[]に格納します。

　（10）と（11）では、このwordに納められた単語をカウントして、頻度数が大きい単語から順に並べ替えています。

In [20]:
#[2]テキスト抽出と出現頻度のカウント手順の設定
def count_word_frequency(pdf_path):
    reader = PdfReader(pdf_path) #(1)PDFファイルの読み込み
    full_text = "" #(2)抽出したテキストを格納する変数の設定
    for page in reader.pages: #(3)1頁ごとの処理実施指示
        text = page.extract_text() #(4)テキストを抽出しtextに保存
        if text:
            full_text += text #(5)新たに抽出したテキストをtextに追加

    t = Tokenizer() #(6)Janomeにより日本語テキストを単語に分解
    words = [] #分解された単語を格納するセルの創設
    stop_words = {"委員", "金融", "2026", "こと"} #(7)カウントから除外する単語の定義

    for token in t.tokenize(full_text): #(8)助詞・記号を除外して名詞のみのカウントを指示
        part_of_speech = token.part_of_speech.split(',')[0]
        if part_of_speech == '名詞': #単語が名詞か否かの判定
            if len(token.surface) > 1: #文字数が2文字以上であるかの判定
                   if token.surface not in stop_words: #除外リストに含まれるか否かの判定
                    words.append(token.surface) #(9)上記3つの「判定」をクリアした単語のみwordに格納

    word_counts = Counter(words)  #(10)word内の出現頻度をカウント
    most_common_words = word_counts.most_common(20)  #(11)出現頻順に並び替え上位20個）
    return most_common_words

　セル[2]の（1）～（11）において設計された作業を次のセル[3]で実行します。(12)では、Google colabの「ファイル」にアップロードした日本銀行金融政策決定会合議事録要旨のpdfファイルを指定して読み込みます。（13）では、セル[2]で設計した、テキストの抽出、テキストの単語への分解、単語から助詞他の除去、除外リストに含まれるか否かの判断を経て、word[]に格納された単語をカウントし、並べ替えます。

　2026年7月30-31日開催と2025年7月30-31日開催の2つの日本銀行金融政策決定会合の議事録要旨を比較すると、前者は「物価」13回（出現頻度1位）、「上昇」12回（同2位）であるのに対し、2025年7月末の会合では「物価」は6回、「上昇」も6回の半数に留まっています。

　今後の研究の方向としては、これらの議事録の出現頻度を時系列的に整理し、実際の政策金利との関係を検証する研究が一つ目のテーマ、そして各国中央銀行ごとに同様のLLM分析を行い、政策金利の変更に「国の固有効果」が存在するかを検証することが、二つ目のテーマとなりそうです。

In [23]:
#[3]PDFファイルの読み込みと出現頻度カウントの実施
if __name__ == "__main__":
    pdf_file_path = "boj_mpm_july2026.pdf" #(12)左の「ファイル」へアップロードしたファイルを指定
    try:
        results = count_word_frequency(pdf_file_path) #(13)テキスト抽出と単語数カウントを実行し結果を保存
        print("\n【日本銀行金融政策決定会合議事録要旨　単語出現頻度ランキング（上位20個）】")
        print("-" * 30)
        for rank, (word, count) in enumerate(results, 1):
            print(f"{rank:>2}位: {word} ({count}回)")

    except FileNotFoundError:
        print(f"エラー①: 指定されたファイル '{pdf_file_path}' が見つかりません。")
    except Exception as e:
        print(f"エラー②（実行中の不具合）が発生しました: {e}")


【日本銀行金融政策決定会合議事録要旨　単語出現頻度ランキング（上位20個）】
------------------------------
 1位: 物価 (13回)
 2位: 上昇 (12回)
 3位: 政策 (11回)
 4位: 企業 (9回)
 5位: 認識 (9回)
 6位: 経済 (8回)
 7位: 程度 (8回)
 8位: 必要 (8回)
 9位: 反対 (8回)
10位: 決定 (7回)
11位: 影響 (7回)
12位: 価格 (7回)
13位: 浅田 (6回)
14位: 10 (6回)
15位: よう (6回)
16位: 環境 (6回)
17位: 会合 (5回)
18位: てい (5回)
19位: 担保 (5回)
20位: コール (5回)
